In [31]:
from numpy import random
from ortools.math_opt.python import mathopt
import pandas as pd

In [32]:
MAX_INTAKE_FACTOR = 1.15
# rng = random.default_rng(42)

The model assumes a monthly MWh procurement model with costs in $/MWh and emissions in Metric tons of CO2e/MWh


Units
- demand: MWh per month
- capacity: MWh per month
- emissions: metric tons CO2e per MWh
- cost: $ per MWh
- carbon intensity CO2e / MWh




In [33]:
# DCs Data

data_centers = pd.DataFrame({
    "data_center": [
        "DC_Texas",
        "DC_Virginia",
        "DC_Iowa",
        "DC_Oregon",
        "DC_Georgia"
    ],
    # Monthly demand in MWh
    "demand_mwh": [
        42000,
        55000,
        30000,
        26000,
        38000
    ],
    # Allowed emissions intensity in metric tons CO2e / MWh
    "carbon_intensity_cap": [
        0.44,
        0.48, 
        0.52, 
        0.50, 
        0.45  
]
})


# Maximum safe energy intake, assumed at 15% above expected demand
# data_centers["max_energy_mwh"] = (data_centers["demand_mwh"] * MAX_INTAKE_FACTOR).round(0).astype(int)

In [34]:
# Providers Data

providers = pd.DataFrame({
    "provider": [
        "Coal_A",
        "Coal_B",
        "Gas_A",
        "Gas_B",
        "Oil_A",
        "Wind_A",
        "Wind_B",
        "Solar_A",
        "Solar_B"
    ],
    "source": [
        "coal",
        "coal",
        "lng",
        "lng",
        "oil",
        "wind",
        "wind",
        "solar",
        "solar"
    ],
    # Base cost in $/MWh
    "base_cost_per_mwh": [
        65,
        72,
        80,
        95,
        140,
        45,
        55,
        40,
        50
    ],
    # Emissions in metric tons CO2e / MWh
    "emissions_tco2e_per_mwh": [
        0.95,
        1.05,
        0.42,
        0.50,
        0.78,
        0.01,
        0.02,
        0.02,
        0.03
    ],
    # Monthly provider capacity in MWh
    "capacity_mwh": [
        60000,
        45000,
        65000,
        45000,
        25000,
        40000,
        35000,
        37000,
        32000
    ]
})

In [35]:
# Cost Parameters

DC = data_centers["data_center"].tolist()  # Data Centers, indexed i
P = providers["provider"].tolist()  # Providers, indexed j

# For each data-center, we include a term in the cost parameter to simulate the differences in costs of providers to supply different localities. This is a simpler stand-in
# for more robust modeling of energy transit costs.

location_adder = {  
    "DC_Texas": 0,
    "DC_Virginia": 8,
    "DC_Iowa": -3,
    "DC_Oregon": 4,
    "DC_Georgia": 6
}

base_cost = providers.set_index("provider")["base_cost_per_mwh"].to_dict()

cost = {}

# For the cost parameter, c_ij, we add our simulated base cost per MWh, 

for dc in DC:
    for p in P:

        cost[(dc, p)] = round(
            base_cost[p]
            + location_adder[dc],
            2
        )

data_centers = data_centers.set_index("data_center")
providers = providers.set_index("provider")


In [36]:
model = mathopt.Model(name="energy_dc_v1")

# DECISION VARIABLES
x = {
    (i, j): model.add_variable(lb=0.0, name=f"x[{i}, {j}]")
    for i in DC
    for j in P
}

In [37]:
# CONSTRAINTS

# The total acquired MWh for Data Center i must be within its demand - max-intake range
demand_cons = [
    model.add_linear_constraint(expr=mathopt.fast_sum(x[i, j] for j in P), lb=data_centers["demand_mwh"][i]) 
    for i in DC
    ]

# The carbon intensity for data center i must not exceed its limit
carbon_compliance_2 = [model.add_linear_constraint(
        expr=mathopt.fast_sum(providers["emissions_tco2e_per_mwh"][j] * x[i, j] for j in P) - (data_centers["carbon_intensity_cap"][i] * mathopt.fast_sum(x[i, j] for j in P)),
        ub=0.0
        )
        for i in DC
        ]

# The total energy procured from supplier j must not exceed its capacity
supplier_cap = [
    model.add_linear_constraint(expr=mathopt.fast_sum([x[i, j] for i in DC]), ub=providers["capacity_mwh"][j])
    for j in P
]

In [38]:
# OBJECTIVE FUNCTION
model.minimize(
    mathopt.fast_sum(cost[i, j] * x[i, j] for i in DC for j in P)
)

In [39]:
# SOLVE
params = mathopt.SolveParameters(enable_output=True)
result = mathopt.solve(model, mathopt.SolverType.GLOP)

if result.termination.reason == mathopt.TerminationReason.OPTIMAL:
    values = result.variable_values()

try:
    print(result.objective_value())
except Exception as e:
    print(e)

10541999.999999998


In [40]:
dc_shadow_prices = pd.DataFrame({
    "data_center": DC,
    "demand_shadow_price": [
        result.dual_values(con) for con in demand_cons
    ],
    "carbon_shadow_price": [
        result.dual_values(con) for con in carbon_compliance_2
    ],
})

provider_shadow_prices = pd.DataFrame({
    "provider": P,
    "capacity_shadow_price": [
        result.dual_values(con) for con in supplier_cap
    ],
})

print("Data center shadow prices")
print(dc_shadow_prices)
print("=====================")
print("Provider shadow prices")
print(provider_shadow_prices)




reduced_costs_df = pd.DataFrame(
    [
        {
            "data_center": i,
            "provider": j,
            "x_mwh": result.variable_values(x[i, j]),
            "reduced_cost": result.reduced_costs(x[i, j]),
        }
        for i in DC
        for j in P
    ]
)

# print("\nReduced costs by data center-provider pair")
# print(reduced_costs_df)

Data center shadow prices
   data_center  demand_shadow_price  carbon_shadow_price
0     DC_Texas                 65.0        -1.478705e-14
1  DC_Virginia                 73.0         0.000000e+00
2      DC_Iowa                 62.0         0.000000e+00
3    DC_Oregon                 69.0         0.000000e+00
4   DC_Georgia                 71.0         0.000000e+00
Provider shadow prices
  provider  capacity_shadow_price
0   Coal_A                    0.0
1   Coal_B                    0.0
2    Gas_A                    0.0
3    Gas_B                    0.0
4    Oil_A                    0.0
5   Wind_A                  -20.0
6   Wind_B                  -10.0
7  Solar_A                  -25.0
8  Solar_B                  -15.0
